# N32 — Agent Pipeline Smoke Test

Validates that every agent in the v0.9 multi-agent system (N25–N31) can be imported and called without errors, using real lap data from the Melbourne 2025 race for driver NOR (McLaren), lap 20.

Each cell calls `debug_agent.py` as a subprocess so output is captured inline and frozen after execution. This means you can run this notebook in CI or share it as evidence that the pipeline is working end-to-end.

**Pass criteria:** every cell exits with code 0, no `[ERROR]` lines in any output.

In [2]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("__file__").resolve().parent.parent.parent
_p = Path(".").resolve()
while _p != _p.parent:
    if (_p / ".git").exists():
        REPO_ROOT = _p
        break
    _p = _p.parent

PYTHON = sys.executable
DEBUG_SCRIPT = str(REPO_ROOT / "scripts" / "debug_agent.py")
CLI_SCRIPT   = str(REPO_ROOT / "scripts" / "run_simulation_cli.py")

PROVIDER = "openai"

# Force UTF-8 in subprocesses so box-drawing chars don't crash on Windows cp1252
_ENV = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1"}

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"Python    : {PYTHON}")
print(f"Provider  : {PROVIDER}")
print(f"debug_agent.py exists : {Path(DEBUG_SCRIPT).exists()}")
print(f"run_simulation_cli.py exists : {Path(CLI_SCRIPT).exists()}")


def run_agent(agent: str, extra: list[str] = []) -> subprocess.CompletedProcess:
    """Call debug_agent.py for the given agent with fixed Melbourne/NOR/McLaren/lap-20 args."""
    cmd = [
        PYTHON, DEBUG_SCRIPT,
        "--agent",    agent,
        "--gp",       "Melbourne",
        "--driver",   "NOR",
        "--team",     "McLaren",
        "--lap",      "20",
        "--provider", PROVIDER,
    ] + extra
    print(f"\n{'='*60}")
    print(f"AGENT: {agent.upper()}")
    print(f"{'='*60}")
    result = subprocess.run(cmd, cwd=str(REPO_ROOT), capture_output=False, env=_ENV)
    print(f"\nExit code: {result.returncode}")
    assert result.returncode == 0, f"{agent} agent exited with code {result.returncode}"
    return result

REPO_ROOT : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
Python    : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\.venv\Scripts\python.exe
Provider  : openai
debug_agent.py exists : True
run_simulation_cli.py exists : True


## N25 — Pace Agent

XGBoost lap-time prediction + bootstrap confidence interval (P10/P90).

In [2]:
run_agent("pace")


AGENT: PACE

Exit code: 0


CompletedProcess(args=['c:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\.venv\\Scripts\\python.exe', 'C:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\scripts\\debug_agent.py', '--agent', 'pace', '--gp', 'Melbourne', '--driver', 'NOR', '--team', 'McLaren', '--lap', '20', '--provider', 'openai'], returncode=0)

## N26 — Tire Agent

TireDegTCN MC Dropout cliff estimation — laps to cliff (P10/P50/P90) and warning level.

In [3]:
run_agent("tire")


AGENT: TIRE

Exit code: 0


CompletedProcess(args=['c:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\.venv\\Scripts\\python.exe', 'C:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\scripts\\debug_agent.py', '--agent', 'tire', '--gp', 'Melbourne', '--driver', 'NOR', '--team', 'McLaren', '--lap', '20', '--provider', 'openai'], returncode=0)

## N27 — Race Situation Agent

LightGBM overtake probability + SC-within-3-laps probability, threat level badge.

In [4]:
run_agent("situation")


AGENT: SITUATION

Exit code: 0


CompletedProcess(args=['c:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\.venv\\Scripts\\python.exe', 'C:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\scripts\\debug_agent.py', '--agent', 'situation', '--gp', 'Melbourne', '--driver', 'NOR', '--team', 'McLaren', '--lap', '20', '--provider', 'openai'], returncode=0)

## N28 — Pit Strategy Agent

Quantile stop duration (P05/P50/P95) + N16 undercut success probability + compound recommendation.

In [5]:
run_agent("pit")


AGENT: PIT

Exit code: 0


CompletedProcess(args=['c:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\.venv\\Scripts\\python.exe', 'C:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\scripts\\debug_agent.py', '--agent', 'pit', '--gp', 'Melbourne', '--driver', 'NOR', '--team', 'McLaren', '--lap', '20', '--provider', 'openai'], returncode=0)

## N29 — Radio Agent

RoBERTa sentiment + SetFit intent + BERT NER + RCM parser. Tested with an explicit radio message to exercise the full NLP path.

In [6]:
run_agent("radio", extra=["--radio", "Box box, tyres gone"])


AGENT: RADIO

Exit code: 0


CompletedProcess(args=['c:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\.venv\\Scripts\\python.exe', 'C:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\scripts\\debug_agent.py', '--agent', 'radio', '--gp', 'Melbourne', '--driver', 'NOR', '--team', 'McLaren', '--lap', '20', '--provider', 'openai', '--radio', 'Box box, tyres gone'], returncode=0)

## N30 — RAG Agent

Qdrant vector retrieval + LLM synthesis for FIA regulation questions.

> **Requires:** `OPENAI_API_KEY` set in `.env` (or switch `PROVIDER = "lmstudio"` in setup cell).

In [8]:
run_agent("rag", extra=["--query", "safety car restart procedure"])


AGENT: RAG

Exit code: 0


CompletedProcess(args=['c:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\.venv\\Scripts\\python.exe', 'C:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\scripts\\debug_agent.py', '--agent', 'rag', '--gp', 'Melbourne', '--driver', 'NOR', '--team', 'McLaren', '--lap', '20', '--provider', 'openai', '--query', 'safety car restart procedure'], returncode=0)

## N31 — Strategy Orchestrator

Full pipeline: all sub-agents → MoE routing → MC simulation → LLM synthesis.

> **Requires:** `OPENAI_API_KEY` set in `.env` (or switch `PROVIDER = "lmstudio"` in setup cell).

In [3]:
run_agent("orchestrator")


AGENT: ORCHESTRATOR

Exit code: 0


CompletedProcess(args=['c:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\.venv\\Scripts\\python.exe', 'C:\\Users\\victo\\Desktop\\Documents\\Cuarto Año\\TFG\\F1_Strat_Manager\\scripts\\debug_agent.py', '--agent', 'orchestrator', '--gp', 'Melbourne', '--driver', 'NOR', '--team', 'McLaren', '--lap', '20', '--provider', 'openai'], returncode=0)

## Full system CLI — laps 1-3, no-LLM mode

Runs the complete RaceReplayEngine → per-lap orchestrator loop for 3 laps without any LLM call. This validates the full data pipeline from raw parquet to StrategyRecommendation.

In [4]:
result = subprocess.run(
    [
        PYTHON, CLI_SCRIPT,
        "Melbourne", "NOR", "McLaren",
        "--laps", "1-3",
        "--no-llm",
    ],
    cwd=str(REPO_ROOT),
    capture_output=False,
    env=_ENV,
)
print(f"\nExit code: {result.returncode}")
assert result.returncode == 0, f"CLI simulation exited with code {result.returncode}"


Exit code: 0


## Pass criteria

- Every cell above exits with code 0 (asserted inline).
- No `[ERROR]` lines appear in any cell output.
- The CLI simulation cell prints the Rich table with 3 rows and `Completed successfully.` at the end.

If any agent fails, re-run `debug_agent.py` manually with `--verbose` to see the full traceback:

```bash
python scripts/debug_agent.py --agent <name> --gp Melbourne --driver NOR --team McLaren --lap 20 --verbose
```